# Vintage Resale Business - Data Analysis (2021-2025)

**Author:** Richard Rowe
**Skills:** Python · pandas · matplotlib · seaborn · pdfplumber · data cleaning
**Github:** [rrowe99](https://github.com/rrowe99)

---

I've been buying and reselling products online since 2021. What started with sneakers, Pokémon cards, and even crypto mining equipment gradually evolved into a vintage clothing and workwear business sold through eBay, Depop, Grailed, Instagram, and in person flea markets throughout New England.

This notebook turns five years of messy raw data (1099-K tax forms, marketplace CSV exports, handwritten market notes, and expense records) into a clean, virtual business analysis.

**Questions this notebook answers:**
1. How did the business grow, and where did the money come from?
2. When do sales  happen? (online seasonality vs. market season)
3. Which sales channel is worth it? (price, fees, and take home by platform)
4. What actually makes money? (top items and 2022 item level ROI)
5. Is the business profitable? (revene vs. expenses, 2024-2025)
6. Data limitations
7. Key takeaways & business implications

## Setup

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# Data parses (from src/parse_data.py)
from parse_data import (
    parse_item_csvs, parse_1099s, 
    build_annual_summary, build_expenses, build_market_events,
    parse_ebay_transactions, parse_depop_sales, build_notable_items
)

# Styling
PALETTE = {
    "ebay":     "#E53238",
    "grailed":   "#1B1B1B",
    "depop":     "#FF2300",
    "market":    "#4A90D9",
    "instagram": "#C13584",
}
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.family": "sans-serif", "font.size": 11})
print("✅ Imports ready")

✅ Imports ready


## Data Sources

The raw data comes from for different source types:
- **CSV exports**: Flyp inventory tracker (2022), eBay Seller Hub, and Depop seller dashboard
- **1099-K PDFS**: IRS tax forms; monthly revenue is extracted with 'pdfplmber' and regular expressions
- **Handwritten market notes**: transcribed by hand into a structured event log
- **Expense summaries**: annual expense breakdowns

A large part of this project was cleaning and standardizing these sources before any analysis could be done. All extraction logic lives in 'src/parse_data.py'

In [2]:
# Load all datasets
items   = parse_item_csvs()            # 2022 item level sales (Flyp)
ebay_tx = parse_ebay_transactions()     # eBay orders 2024-2025
depop_tx = parse_depop_sales()          # Depop sales 2023-2025
monthly = parse_1099s()                 # monthly 1099-K revenue by platform
annual = build_annual_summary()         # annual totals by platform
expenses = build_expenses()             # expense categories by year
markets = build_market_events()         # in person market events

print(f"\nItems (2022):     {len(items)} sold items")
print(f"eBay transactions:  {len(ebay_tx)} orders | avg ${ebay_tx['revenue'].mean():.2f}")
print(f"Depop transactions: {len(depop_tx)} sales | avg ${depop_tx['revenue'].mean():.2f}")
print(f"Monthly revenue:    {len(monthly)} platform-months")
print(f"Annual summary:     {annual['year'].nunique()} years, {annual['platform'].nunique()} platforms")
print(f"Market events:      {len(markets)} events")

  ✓ Flyp CSVs: 59 sold items
  ✓ Transaction_report_20240101_20240331.csv: 11 orders
  ✓ Transaction_report_20240401_20240630.csv: 33 orders
  ✓ Transaction_report_20240701_20240930.csv: 27 orders
  ✓ Transaction_report_20241001_20241231.csv: 20 orders
  ✓ Transaction_report_20250101_20250331.csv: 36 orders
  ✓ Transaction_report_20250401_20250630.csv: 25 orders
  ✓ Transaction_report_20250701_20250930.csv: 38 orders
  → eBay total: 190 orders
  ✓ Depop Sales 02_01_2023 - 04_01_2023.csv: 13 sales
  ✓ Depop Sales 03_01_2024 - 05_29_2024.csv: 94 sales
  ✓ Depop Sales 04_01_2023 - 06_30_2023.csv: 23 sales
  ✓ Depop Sales 06_01_2024 - 08_31_2024.csv: 53 sales
  ✓ Depop Sales 07_01_2023 - 09_30_2023.csv: 20 sales
  ✓ Depop Sales 09_01_2024 - 11_30_2024.csv: 11 sales
  ✓ Depop Sales 10_01_2023 - 12_30_2023.csv: 77 sales
  ✓ Depop Sales 12_01_2023 - 02_29_2024.csv: 73 sales
  ✓ Depop Sales 12_01_2024 - 02_28_2025.csv: 14 sales
  → Depop total: 378 sales
 ✓ Paypal_1099K_2022.pdf: 11 months
 ✓ 

In [3]:
# Preview item level data
items[["name", "platform", "cog", "revenue", "profit", "roi_pct"]].head(8).round(2)

,name,platform,cog,revenue,profit,roi_pct
0,Nike Dunk HIgh Pistachio,grailed,300.0,286.20,-13.80,-4.60
1,Vintage Bjork Tee Shirt Reprint,grailed,20.0,36.51,16.51,82.55
2,adidas Ozweego 3 Raf Simons Burgundy,grailed,200.0,205.63,5.63,2.81
3,adidas Yeezy Boost 380 Alien,grailed,100.0,100.64,0.64,0.64
4,I Love Creative Women T Shirt,grailed,30.0,37.52,7.52,25.07
5,1998 Helmut Lang Denim,grailed,110.0,109.02,-0.98,-0.89
6,Sp5der Worldwide Red Angel Number 555 Hoodie Red,grailed,175.0,214.64,39.64,22.65
7,San Juan Denim Atlanta Braves hat,grailed,35.0,37.29,2.29,6.54


In [4]:
# Annual totals at a glance
summary = annual.groupby("year")["gross_revenue"].sum().reset_index()
summary.columns = ["Year", "Gross Revenue"]
summary["YoY Growth"] = summary["Gross Revenue"].pct_change().mul(100).round(1).astype(str) + "%"
summary["Gross Revenue"] = summary["Gross Revenue"].map("${:,.2f}".format)
print("=== Annual Revenue Summary ===")
display(summary)

=== Annual Revenue Summary ===


,Year,Gross Revenue,YoY Growth
0,2021,"$9,149.72",NaN
1,2022,"$8,265.94",-9.7%
2,2023,"$19,963.31",141.5%
3,2024,"$36,274.12",81.7%
4,2025,"$31,021.61",-14.5%
